# 02 - MLP (neural network)
A multi-layer perceptron: only fully connected layers, from 784 pixels to 10 digits.

## 1. Imports and settings

In [1]:
import copy
import json
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from torchvision import datasets

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
torch.manual_seed(SEED)

## 2. Load the data
Download MNIST (first run only) and scale the pixels to 0-1.

In [2]:
train = datasets.MNIST(ROOT / "data", train=True, download=True)
test = datasets.MNIST(ROOT / "data", train=False, download=True)
x_all, y_all = train.data.numpy().astype(np.float32) / 255, train.targets.numpy()
x_test, y_test = test.data.numpy().astype(np.float32) / 255, test.targets.numpy()

## 3. Split the data
54,000 images for training and 6,000 for validation (to compare settings). The 10,000 test images are kept for the end.

In [3]:
x_train, x_val, y_train, y_val = train_test_split(x_all, y_all, test_size=6000, stratify=y_all, random_state=SEED)

## 4. Flatten the images
The MLP reads each 28x28 image as a row of 784 numbers. Convert everything to PyTorch tensors.

In [4]:
x_train, x_val, x_test = (torch.from_numpy(a.reshape(len(a), -1)) for a in (x_train, x_val, x_test))
y_train, y_val, y_test = (torch.from_numpy(a) for a in (y_train, y_val, y_test))

## 5. The model
Each hidden layer is Linear, ReLU, Dropout (dropout randomly switches off neurons while training to reduce overfitting).

In [5]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout):
        super().__init__()
        layers, n_in = [], 28 * 28
        for h in hidden_sizes:
            layers += [nn.Linear(n_in, h), nn.ReLU(), nn.Dropout(dropout)]
            n_in = h
        layers.append(nn.Linear(n_in, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## Helper: accuracy
The share of images the model gets right.

In [6]:
def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        correct = sum((model(x[i:i + 2000]).argmax(1) == y[i:i + 2000]).sum().item() for i in range(0, len(x), 2000))
    return correct / len(x)

## Helper: train
Trains with Adam, checks the validation accuracy after every epoch, and keeps the weights of the best epoch.

In [7]:
def train(model, lr, epochs, batch_size=128):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_acc, best_state = 0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x_train))
        for i in range(0, len(x_train), batch_size):
            batch = order[i:i + batch_size]
            opt.zero_grad()
            loss_fn(model(x_train[batch]), y_train[batch]).backward()
            opt.step()
        val_acc = accuracy(model, x_val, y_val)
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return best_acc

## 6. Find good settings
Try 3 layer layouts and 2 learning rates (5 epochs each) and keep the best one on the validation data.

In [8]:
results = []
for hidden in [(128,), (256, 128), (512, 256)]:
    for lr in [1e-2, 1e-3]:
        val_acc = train(MLP(hidden, dropout=0.3), lr, epochs=5)
        results.append((val_acc, hidden, lr))
        print(f"layers={hidden}  learning rate={lr}  validation accuracy {val_acc:.4f}")
best_acc, best_hidden, best_lr = max(results)
print("best:", best_hidden, best_lr)

layers=(128,)  learning rate=0.01  validation accuracy 0.9685


layers=(128,)  learning rate=0.001  validation accuracy 0.9670


layers=(256, 128)  learning rate=0.01  validation accuracy 0.9647


layers=(256, 128)  learning rate=0.001  validation accuracy 0.9762


layers=(512, 256)  learning rate=0.01  validation accuracy 0.9610


layers=(512, 256)  learning rate=0.001  validation accuracy 0.9782
best: (512, 256) 0.001


## 7. Train the final model
Same idea with the best settings, for more epochs.

In [9]:
model = MLP(best_hidden, dropout=0.3)
start = time.time()
train(model, best_lr, epochs=20)
train_time = time.time() - start
print(f"training time: {train_time:.0f} s")

training time: 16 s


## 8. Check the accuracy
Test accuracy is the main result. Train accuracy shows how much the model overfits.

In [10]:
test_acc = accuracy(model, x_test, y_test)
train_acc = accuracy(model, x_train[:10000], y_train[:10000])
print(f"test accuracy {test_acc:.4f}   train accuracy {train_acc:.4f}")

test accuracy 0.9836   train accuracy 0.9977


## 9. Save the model
The web app loads this file, so it also needs the layer sizes to rebuild the network.

In [11]:
torch.save({"arch": "mlp", "config": dict(hidden_sizes=list(best_hidden), dropout=0.3), "state_dict": model.state_dict()},
           ROOT / "app" / "models" / "mlp_model.pt")

## 10. Save the results
The comparison notebook reads this file.

In [12]:
results = dict(test_acc=test_acc, train_acc=train_acc, train_time=train_time,
               parameters=sum(p.numel() for p in model.parameters()),
               settings=dict(hidden_sizes=list(best_hidden), learning_rate=best_lr, dropout=0.3))
(ROOT / "artifacts" / "mlp_results.json").write_text(json.dumps(results, indent=2))

225